# eCommerce Funnel Analysis
**Goal:** Calculate unique user conversion rates from View -> Intent -> Purchase.

### How to Use This Pipeline
This notebook is designed as an automated ETL pipeline. 
1. Place the new month's raw CSV file into the `data/raw/` folder.
2. In the second code cell, update the `pd.read_csv()` path to point to the new file.
3. Click **Restart Kernel and Run All Cells**.
4. The calculated funnel CSVs will automatically update in the `data/processed/` folder.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("../data/raw/2019-Oct.csv")
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


## Step 1: Raw Event Frequencies
First, looking at the total number of actions taken across the platform. However, these are total events, not unique users.

In [3]:
event_count = df['event_type'].value_counts()
event_count

event_type
view        40779399
cart          926516
purchase      742849
Name: count, dtype: int64

## Step 2: Unique Users per Event
To build an accurate funnel, we calculate the distinct number of users at each stage. Notice that 'purchase' is higher than 'cart', indicating a direct checkout feature.

In [4]:
distinct_event_count = df.groupby('event_type')['user_id'].nunique()
distinct_event_count

event_type
cart         337117
purchase     347118
view        3022130
Name: user_id, dtype: int64

## Step 3: Funnel Stage Deduplication (The Intent Bucket)
Because users can bypass the cart, we create a unified 'Intent' stage (combining 'cart' and 'purchase'). We use `.nunique()` on this subset to ensure users who both carted and purchased are not counted twice.

In [5]:
intent_data = df[df['event_type'].isin(['cart','purchase'])]
true_intent_users = intent_data['user_id'].nunique()
true_intent_users


481458

## Step 4: Dynamic Funnel Aggregation
Finally, assembling these deduplicated metrics into a clean, automated summary table for dashboard ingestion.

In [6]:
data = {
    "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
    "Unique_Users": [
        distinct_event_count['view'],      
        true_intent_users,                 
        distinct_event_count['purchase']   
    ]
}

funnel_df = pd.DataFrame(data)
funnel_df

,Funnel_Stage,Unique_Users
0,01_View,3022130
1,02_Intent,481458
2,03_Purchase,347118


## Step 5: Conversion and Drop-off Analysis
To fulfill the project requirements, we calculate the stage-to-stage conversion and drop-off rates using vectorized division and the `.shift()` method to align our rows.


In [7]:
funnel_df['conversion_rate%'] = (funnel_df['Unique_Users'] / funnel_df['Unique_Users'].shift(1)) * 100 
funnel_df['conversion_rate%'] = funnel_df['conversion_rate%'].fillna(100).round(2)
funnel_df['drop_off_rate%'] = 100 - funnel_df['conversion_rate%']  
funnel_df

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,3022130,100.00,0.00
1,02_Intent,481458,15.93,84.07
2,03_Purchase,347118,72.10,27.90


### Key Insights & Business Recommendations:
* **The Massive Top-of-Funnel Leak:** There is an **84.07% drop-off rate** between viewing a product and showing intent to buy. The vast majority of traffic bounces after seeing the product page. 
    * **Recommendation:** The marketing and UI teams need to investigate the product pages. Consider A/B testing better product images, clearer pricing, or making the "Add to Cart" button more prominent to reduce friction.
* **The High-Intent Goldmine:** Once a user shows intent (cart or direct checkout), the conversion rate to purchase is a highly effective **72.10%**. 
    * **Recommendation:** The checkout pipeline is highly optimized. Marketing spend should be heavily focused on top-of-funnel engagement, as users who reach the middle of the funnel consistently convert.

## Step 6: Scaling Analysis with Functions (DRY Principle)
To compare performance across different segments (e.g., brands), we wrap our funnel logic into a reusable Python function. This prevents code duplication and allows us to generate custom funnels dynamically.

In [8]:
def analyze_brand_funnel(brand_name):
    # 1. Isolating the brand
    brand_df = df[df['brand'] == brand_name]
    
    # 2. Getting distinct counts per stage
    distinct_counts = brand_df.groupby('event_type')['user_id'].nunique()
    
    # Calculating deduplicated intent
    true_intent = brand_df[brand_df['event_type'].isin(['cart', 'purchase'])]['user_id'].nunique()
    
    # 4. dataframe
    data = {
        "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
        "Unique_Users": [
            distinct_counts.get('view', 0),    
            true_intent,
            distinct_counts.get('purchase', 0) 
        ]
    }
    
    brand_funnel_df = pd.DataFrame(data)
    
    # 5. Adding the columns
    brand_funnel_df['conversion_rate%'] = (brand_funnel_df['Unique_Users'] / brand_funnel_df['Unique_Users'].shift(1)) * 100
    brand_funnel_df['conversion_rate%'] = brand_funnel_df['conversion_rate%'].fillna(100).round(2)
    brand_funnel_df['drop_off_rate%'] = 100 - brand_funnel_df['conversion_rate%']
    
    return brand_funnel_df

## Step 7: Brand Comparison (Apple vs. Samsung)
Testing our function on the two highest-volume electronics brands. 

**Key Business Insight:** 
* Samsung continues to outperform Apple at the top of the funnel (**15.06%** vs **13.72%** View-to-Intent conversion). 
* Both brands maintain strong checkout pipelines, but Samsung edges out Apple in final closing power (**66.76%** vs **65.55%** Intent-to-Purchase conversion).

In [9]:
analyze_brand_funnel('apple')

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,729319,100.00,0.00
1,02_Intent,100061,13.72,86.28
2,03_Purchase,65593,65.55,34.45


In [10]:
analyze_brand_funnel('samsung')

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,935464,100.00,0.00
1,02_Intent,140911,15.06,84.94
2,03_Purchase,94073,66.76,33.24


## Step 8: Automated Top 10 Brand Pipeline (Dashboard Prep)
Hardcoding brand names does not scale for monthly reporting. To prepare this data for a BI dashboard (like Power BI or Tableau), I programmatically identified the highest-volume brands and format their funnels into a single, stacked table ("Long Data").

**The Pipeline Logic:**
1. **Dynamic Identification:** We calculate the top 10 brands by total event volume.
2. **Loop & Append:** We loop each top brand through our `analyze_brand_funnel` function, inserting a `Brand` column so the BI tool can filter the data.
3. **Stacking (Concat):** We combine all 10 individual tables into one master dataframe (`master_brand_df`).
4. **Export:** Both the overall funnel and the top 10 brand comparison tables are exported as clean CSVs to the `data/processed/` directory, ready for dashboard ingestion.

In [11]:
#Exporting the overall funnel
funnel_df.to_csv('../data/processed/01_overall_funnel.csv', index=False)

# Finding the top 10 brands dynamically based on who has the most rows
top_10_brands = df['brand'].value_counts().head(10).index.tolist()

# Createing an empty list to hold our tables
all_brand_funnels = []

# 4. Loop through the top 10 brands
for brand in top_10_brands:
    # Run function
    temp_df = analyze_brand_funnel(brand)
    
    # Add a column so we know which brand
    temp_df.insert(0, 'Brand', brand) 
    
    # Add this brand's table to our list
    all_brand_funnels.append(temp_df)

# 5. Stack all 10 tables on top of each other into one master dataframe
master_brand_df = pd.concat(all_brand_funnels, ignore_index=True)

# 6. Export the final master brand table to CSV!
master_brand_df.to_csv('../data/processed/02_brand_comparison.csv', index=False)

# Display the master table to check our work
master_brand_df

,Brand,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,samsung,01_View,935464,100.00,0.00
1,samsung,02_Intent,140911,15.06,84.94
2,samsung,03_Purchase,94073,66.76,33.24
3,apple,01_View,729319,100.00,0.00
4,apple,02_Intent,100061,13.72,86.28
5,apple,03_Purchase,65593,65.55,34.45
6,xiaomi,01_View,470832,100.00,0.00
7,xiaomi,02_Intent,55456,11.78,88.22
8,xiaomi,03_Purchase,35063,63.23,36.77
9,huawei,01_View,234061,100.00,0.00
